# Can we predict next-month churn?
Target: `churn_next_month` (1 = account cancels in the month after the snapshot). Rows are monthly account snapshots.

Plan: audit the data (including leakage), build features available at snapshot time, evaluate with a **time-based split** (train on the past, test on the future) and group-aware CV, and compare against baselines.

In [ ]:
import os; os.environ['LOKY_MAX_CPU_COUNT'] = '4'
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve, brier_score_loss
from sklearn.inspection import permutation_importance
pd.set_option('display.width', 140)
df = pd.read_csv('data.csv')
print(df.shape, '| accounts:', df.account_ref.nunique(), '| churn rate: %.3f' % df.churn_next_month.mean())
print(df.isna().sum().sum(), 'missing;', df.duplicated(['account_ref', 'snapshot_month']).sum(), 'duplicate account-months')
df.describe().T

## 1. Leakage check: `lifetime_months`
The dictionary says it is the *total months the account was active, from a CRM export of 2026-01-05* - i.e. measured after the snapshot, so it contains future information.

In [ ]:
gap = df.lifetime_months - df.tenure_months
print(gap.groupby(df.churn_next_month).describe())
print('Churners with lifetime == tenure:', (gap[df.churn_next_month == 1] == 0).mean())

For every churned row `lifetime_months == tenure_months` (the account's life ends at the snapshot), while non-churners have a longer lifetime. The column effectively encodes the label and would not be known when making a real prediction. **It is excluded from all models.** Section 4 quantifies how misleading it would be.

In [ ]:
df['month'] = pd.PeriodIndex(df.snapshot_month, freq='M')
df = df.sort_values(['account_ref', 'month']).reset_index(drop=True)
# Features use only information available at the snapshot (current + earlier rows of the same account)
g = df.groupby('account_ref')
df['plan_code'] = df.plan.map({'basic': 0, 'pro': 1, 'team': 2})
df['fee_per_seat'] = df.monthly_fee / df.seats
df['usage_per_seat'] = df.usage_hours / df.seats
df['usage_change'] = df.usage_hours - g.usage_hours.shift(1)
df['usage_ma3'] = g.usage_hours.transform(lambda s: s.rolling(3, min_periods=1).mean())
df['usage_vs_ma3'] = df.usage_hours - df.usage_ma3
df['tickets_3m'] = g.support_tickets.transform(lambda s: s.rolling(3, min_periods=1).sum())
df['seat_change'] = df.seats - g.seats.shift(1)
df['fee_change'] = df.monthly_fee - g.monthly_fee.shift(1)
prev_plan = g.plan.shift(1)
df['plan_change'] = (prev_plan.notna() & (df.plan != prev_plan)).astype(int)
BASE = ['plan_code', 'seats', 'monthly_fee', 'tenure_months', 'usage_hours', 'support_tickets']
ENG = BASE + ['fee_per_seat', 'usage_per_seat', 'usage_change', 'usage_ma3', 'usage_vs_ma3',
              'tickets_3m', 'seat_change', 'fee_change', 'plan_change']
y = df.churn_next_month

## 2. Exploration

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(17, 3.5))
df.groupby('plan').churn_next_month.mean().plot.bar(ax=ax[0], title='Churn rate by plan', color='#4c78a8')
df.groupby(df.tenure_months.clip(upper=12)).churn_next_month.mean().plot(ax=ax[1], marker='o', title='Churn rate by tenure (12 = 12+)')
df.groupby(pd.qcut(df.usage_hours, 10, duplicates='drop'), observed=True).churn_next_month.mean().plot(ax=ax[2], marker='o', title='Churn rate by usage decile')
ax[2].set_xticklabels([])
df.groupby('month').churn_next_month.mean().plot(ax=ax[3], title='Churn rate by snapshot month')
plt.tight_layout(); plt.show()
df.groupby('support_tickets').churn_next_month.agg(['mean', 'size']).T

## 3. Evaluation setup
The model would score *future* months, so the main test is temporal: train on snapshots up to 2025-04, test on 2025-06 onward (the one-month gap avoids label overlap, since the label looks one month ahead). Metrics: ROC AUC, PR AUC (average precision; no-skill = churn rate) and Brier score.

In [ ]:
cut_train, cut_test = pd.Period('2025-04', 'M'), pd.Period('2025-06', 'M')
tr, te = df[df.month <= cut_train], df[df.month >= cut_test]
print(len(tr), len(te), 'churn rate train %.3f test %.3f' % (tr.churn_next_month.mean(), te.churn_next_month.mean()))
print('Test accounts also seen in train: %d of %d' % (te.account_ref[te.account_ref.isin(tr.account_ref)].nunique(), te.account_ref.nunique()))

def models():
    gb = lambda: HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, max_iter=300, random_state=0)
    lr = lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    return {
        'logistic (raw features)': (lr(), BASE),
        'logistic (engineered)': (lr(), ENG),
        'grad. boosting (raw)': (gb(), BASE),
        'grad. boosting (engineered)': (gb(), ENG),
    }

def prep(X, name):  # logistic regression needs no NaNs: first-month history features -> 0 change
    return X.fillna(0) if 'logistic' in name else X

yt = te.churn_next_month.values
base_rate = yt.mean()
rows, preds = [], {}
for name, (m, cols) in models().items():
    m.fit(prep(tr[cols], name), tr.churn_next_month)
    p = m.predict_proba(prep(te[cols], name))[:, 1]
    preds[name] = p
    rows.append((name, roc_auc_score(yt, p), average_precision_score(yt, p), brier_score_loss(yt, p)))
res = pd.DataFrame(rows, columns=['model', 'ROC AUC', 'PR AUC', 'Brier']).set_index('model')
res.loc['(no skill)'] = [0.5, base_rate, base_rate * (1 - base_rate)]
res.round(3)

## 4. How much would the leaky column inflate results?

In [ ]:
m = HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, max_iter=300, random_state=0)
cols = ENG + ['lifetime_months']
m.fit(tr[cols], tr.churn_next_month); p = m.predict_proba(te[cols])[:, 1]
best = preds['grad. boosting (engineered)']
print('WITH lifetime_months -> ROC AUC %.3f, PR AUC %.3f  (not achievable in production)' % (roc_auc_score(yt, p), average_precision_score(yt, p)))
print('WITHOUT (honest)     -> ROC AUC %.3f, PR AUC %.3f' % (roc_auc_score(yt, best), average_precision_score(yt, best)))

## 5. Robustness: grouped cross-validation
The same account never appears in both train and validation folds (rows of one account are highly correlated).

In [ ]:
out = []
for name in ['logistic (engineered)', 'grad. boosting (engineered)']:
    m, cols = models()[name]; s = []
    for a, b in GroupKFold(n_splits=5).split(df, y, df.account_ref):
        m.fit(prep(df.loc[a, cols], name), y[a])
        p = m.predict_proba(prep(df.loc[b, cols], name))[:, 1]
        s.append((roc_auc_score(y[b], p), average_precision_score(y[b], p)))
    s = np.array(s)
    out.append((name, s[:, 0].mean(), s[:, 0].std(), s[:, 1].mean(), s[:, 1].std()))
pd.DataFrame(out, columns=['model', 'AUC mean', 'AUC sd', 'PR AUC mean', 'PR AUC sd']).set_index('model').round(3)

## 6. Practical view: ranking accounts (temporal test set)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
for name, p in preds.items():
    pr, rc, _ = precision_recall_curve(yt, p); ax[0].plot(rc, pr, label=name)
ax[0].axhline(base_rate, ls='--', c='grey')
ax[0].set_xlabel('recall'); ax[0].set_ylabel('precision'); ax[0].legend(fontsize=7); ax[0].set_title('Precision-recall (test)')
order = np.argsort(-best)
dec = pd.qcut(pd.Series(best).rank(method='first', ascending=False), 10, labels=range(1, 11))
lift = pd.Series(yt).groupby(dec.values, observed=True).mean() / base_rate
lift.plot.bar(ax=ax[1], color='#4c78a8'); ax[1].axhline(1, c='grey', ls='--')
ax[1].set_title('Lift by predicted-risk decile (1 = riskiest)')
plt.tight_layout(); plt.show()
top = order[:int(0.1 * len(yt))]
print('Riskiest 10%% of snapshots: precision %.2f (base rate %.2f), captures %.0f%% of churners' % (yt[top].mean(), base_rate, 100 * yt[top].sum() / yt.sum()))

## 6b. The spike in churn (Nov 2024 - Jan 2025)
Monthly churn jumps from ~4-10% to 40-55% for snapshots in Nov 2024 - Jan 2025 (see the chart in section 2). Plan mix is stable, so this is a time-driven event (pricing change, outage, seasonality, ...) that the available columns do not explain. Two consequences: (a) pooled CV metrics mix two regimes, and (b) a model cannot anticipate such an event. Below: is there signal *within* each regime?

In [ ]:
df['regime'] = np.where(df.month.between(pd.Period('2024-11', 'M'), pd.Period('2025-01', 'M')), 'spike', 'normal')
m, cols = models()['grad. boosting (engineered)']
oof = np.zeros(len(df))
for a, b in GroupKFold(n_splits=5).split(df, y, df.account_ref):
    m.fit(df.loc[a, cols], y[a]); oof[b] = m.predict_proba(df.loc[b, cols])[:, 1]
df['oof'] = oof
rep = df.groupby('regime').apply(lambda d: pd.Series({'rows': len(d), 'churn rate': d.churn_next_month.mean(),
        'ROC AUC': roc_auc_score(d.churn_next_month, d.oof), 'PR AUC': average_precision_score(d.churn_next_month, d.oof)}), include_groups=False)
print('Pooled out-of-fold AUC: %.3f' % roc_auc_score(y, oof))
rep.round(3)

## 7. What drives predictions? (permutation importance, test set)

In [ ]:
m, cols = models()['grad. boosting (engineered)']
m.fit(tr[cols], tr.churn_next_month)
pi = permutation_importance(m, te[cols], te.churn_next_month, scoring='roc_auc', n_repeats=5, random_state=0)
pd.Series(pi.importances_mean, index=cols).sort_values().plot.barh(title='Drop in test AUC when feature is shuffled', figsize=(6, 4))
plt.show()

## Conclusion
**Can we predict churn? Partly - there is real but modest signal.**

- **Leakage:** `lifetime_months` is measured after the snapshot (CRM export of 2026-01-05) and equals `tenure_months` for every churner. Using it gives AUC ~0.996, which is meaningless for a real forecast. It was excluded.
- **Honest performance:** with a time-based split (train to 2025-04, test 2025-06+) ROC AUC is ~0.66-0.69 and PR AUC ~0.20-0.21 against a 0.09 base rate (about 2x lift). Grouped CV gives AUC ~0.68-0.71. The riskiest 10% of snapshots churn at ~23% versus 9% overall and contain ~25% of churners.
- **Drivers:** `support_tickets` is the strongest signal (churn rises from ~10% with 0 tickets to ~25% with 2 and ~44% with 4) and `usage_hours` is second (lower usage means higher churn). Plan, seats, fee and tenure add almost nothing. Engineered history features (usage change, rolling tickets) did not help, and the gradient-boosting model did not beat plain logistic regression on the temporal test.
- **Caveat - time regime:** snapshots from Nov 2024 to Jan 2025 have 40-55% churn versus ~10% otherwise. No column explains this, so a model cannot anticipate such an event, and within the spike it ranks accounts less well (AUC ~0.64). Test-period churn (9%) is also much lower than the training average (19%), so predicted probabilities are not calibrated for the test period - use the model to *rank* accounts, and recalibrate on recent data before using probabilities.
- **Suggested next steps:** find what happened in Nov 2024 - Jan 2025; get richer behavioural data (logins, feature use, billing or payment failures, NPS, contract renewal dates); define a retention-offer budget and choose a threshold from the precision/recall trade-off.
- **Execution note:** the cells were run in-process rather than with a Jupyter kernel, because the sandbox blocks local ports. Re-running in Jupyter should give the same results.